# NOVAMART — Phase 7 Inventory Planning

This is a **scenario policy**, not a deployed ERP reorder system. Public inventory is publisher-generated synthetic data; supplier lead times and costs are additional simulations. Forecasts come from Phase 6.

## 1. Inputs, lineage, and assumptions
Read `reports/INVENTORY_ASSUMPTIONS.md` before interpreting results. Latest snapshot date is 2024-01-30. Service target 95% (z 1.645), ordering cost 75, holding rate 20%, and 14-day review interval are assumptions.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd() if (Path.cwd()/"data/processed/daily_operations.csv").exists() else Path.cwd().parent
assert (ROOT/"reports/demand_forecast_14d_summary.csv").exists()
sys.path.insert(0,str(ROOT/"src"))
from inventory_optimization import build
summary=build()
rows=pd.read_csv(ROOT/"reports/replenishment_recommendations.csv")
print(summary)
display(rows[["local_sku_id","inventory_origin","demand_origin","supplier_cost_origin","policy_origin"]].head())

## 2. Forecast demand rate and simulated lead time
The 14-day forecast sum / 14 remains fractional. Expected lead-time demand is daily forecast × supplier expected days. The supplier field is simulated; source inventory remains a snapshot proxy.

In [ ]:
display(rows[["local_sku_id","forecast_14d_units","avg_forecast_daily_units","expected_lead_time_days","lead_time_demand"]].head())

## 3. Safety stock, reorder point, and EOQ
Safety stock = 1.645 × out-of-sample daily residual SD × sqrt(lead days). ROP = lead-time demand + safety stock. EOQ is a separate reference using annualized demand and assumed costs; no final order simply equals EOQ.

In [ ]:
display(rows[["local_sku_id","forecast_error_std","safety_stock","reorder_point","current_inventory","eoq_reference"]].head())
display(pd.read_csv(ROOT/"reports/eoq_sensitivity.csv").round(2))
display(Image(filename=str(ROOT/"images/inventory/06_eoq_sensitivity.png")))

## 4. Status, recommended quantity, and management exceptions
Only inventory at or below ROP triggers a proposed order. Target stock covers lead time plus 14 review days and safety stock; final physical quantity is the ceiling of the gap. Low-stock and excess screens are review flags.

In [ ]:
display(rows.groupby(["inventory_status","recommendation_priority"]).agg(local_skus=("local_sku_id","size"),proposed_units=("recommended_order_quantity","sum")))
display(rows[rows.local_sku_id.isin(["S001_P0001","S001_P0003","S001_P0010"])][["local_sku_id","forecast_14d_units","avg_forecast_daily_units","lead_time_demand","forecast_error_std","safety_stock","reorder_point","current_inventory","eoq_reference","target_stock","recommended_order_quantity","inventory_status"]].round(2))
for name in ["01_inventory_status.png","02_inventory_vs_reorder.png","03_safety_stock_category.png","04_coverage_distribution.png","05_top_orders.png"]:display(Image(filename=str(ROOT/"images/inventory"/name)))

## 5. Business interpretation and limits
97 reorder flags reflect the source snapshot, simulated lead time, and assumed service buffer; they do not establish a real shortage. The source ledger and simulated procurement cannot be reconciled. Recommendations require planner review, open-order data, packs, budget, and a validated service policy before use. No ABC analysis or supplier scoring is performed here.

In [ ]:
assert len(rows)==100 and rows.local_sku_id.nunique()==100
assert rows.recommended_order_quantity.ge(0).all()
assert rows.recommended_order_quantity.sum()==summary["recommended_units"]
print("Notebook assertions passed.")